In [1]:
from pathlib import Path

import numpy as np
import pandas as pd



# 1 Define input and output files


BASE_DIR = Path.cwd()

INPUT_PATH = (
    BASE_DIR
    / "clear_text_states_phase9_rerun.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "phase10_synthetic_learner_environment_rerun.csv"
)

print(
    "Input file exists:",
    INPUT_PATH.exists()
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Phase 9 output not found: {INPUT_PATH}"
    )


# 2 Load the Phase 9 text-state dataset


state_df = pd.read_csv(
    INPUT_PATH
)

required_columns = [
    "Difficulty_D",
    "Cognitive_Effort_C",
    "D_level",
    "C_level",
    "D_label",
    "C_label"
]

missing_columns = [
    column
    for column in required_columns
    if column not in state_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )

print(
    "\nShape:",
    state_df.shape
)

print(
    "\nInput sample:"
)

print(
    state_df[
        [
            "Difficulty_D",
            "Cognitive_Effort_C",
            "D_level",
            "C_level"
        ]
    ].head()
)



# 3 Define the three simulated learner profiles


LEARNER_PROFILES = {
    "Struggling": 0.30,
    "Average": 0.55,
    "Strong": 0.80
}

print(
    "\nLearner profiles:"
)

print(
    LEARNER_PROFILES
)



# 4 Define the simulated probability of a correct response


def sigmoid(
    value
):
    return (
        1
        / (
            1
            + np.exp(
                -value
            )
        )
    )


def probability_correct(
    ability,
    difficulty,
    cognitive_effort,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):
    score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    return sigmoid(
        score
    )



# 5 Check the response model on example learner-text cases


examples = [
    {
        "name": "Strong / low difficulty and effort",
        "ability": 0.80,
        "D": 0.20,
        "C": 0.20
    },
    {
        "name": "Average / medium difficulty and effort",
        "ability": 0.55,
        "D": 0.50,
        "C": 0.50
    },
    {
        "name": "Struggling / high difficulty and effort",
        "ability": 0.30,
        "D": 0.80,
        "C": 0.80
    }
]

print(
    "\nResponse-model sanity check:"
)

for example in examples:

    p_correct = probability_correct(
        ability=example[
            "ability"
        ],
        difficulty=example[
            "D"
        ],
        cognitive_effort=example[
            "C"
        ]
    )

    print(
        example["name"],
        "P(correct) =",
        round(
            p_correct,
            3
        )
    )



# 6 Generate learner-specific correctness probabilities


learner_frames = []

for (
    learner_name,
    ability
) in LEARNER_PROFILES.items():

    learner_data = (
        state_df.copy()
    )

    learner_data[
        "Learner_Profile"
    ] = learner_name

    learner_data[
        "Learner_Ability"
    ] = ability

    learner_data[
        "P_correct"
    ] = learner_data.apply(
        lambda row:
            probability_correct(
                ability=ability,
                difficulty=row[
                    "Difficulty_D"
                ],
                cognitive_effort=row[
                    "Cognitive_Effort_C"
                ]
            ),
        axis=1
    )

    learner_frames.append(
        learner_data
    )


simulation_df = pd.concat(
    learner_frames,
    ignore_index=True
)

print(
    "\nSimulation shape:",
    simulation_df.shape
)

print(
    "\nP(correct) by learner profile:"
)

print(
    simulation_df
    .groupby(
        "Learner_Profile"
    )[
        "P_correct"
    ]
    .describe()
)



# 7 Sample synthetic correct and incorrect responses


RANDOM_SEED = 42

rng = np.random.default_rng(
    RANDOM_SEED
)

simulation_df[
    "Correct"
] = rng.binomial(
    n=1,
    p=simulation_df[
        "P_correct"
    ]
)

print(
    "\nMean simulated correctness:"
)

print(
    simulation_df
    .groupby(
        "Learner_Profile"
    )[
        "Correct"
    ]
    .mean()
)



# 8 Calculate initial recent learner performance P


simulation_df = (
    simulation_df
    .sort_values(
        "Learner_Profile",
        kind="stable"
    )
    .reset_index(
        drop=True
    )
)

simulation_df[
    "Performance_P"
] = (
    simulation_df
    .groupby(
        "Learner_Profile",
        sort=False
    )[
        "Correct"
    ]
    .transform(
        lambda responses:
            responses
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
    )
)

print(
    "\nPerformance P distribution:"
)

print(
    simulation_df
    .groupby(
        "Learner_Profile"
    )[
        "Performance_P"
    ]
    .describe()
)



# 9 Discretise learner performance into three levels


def performance_level(
    performance
):
    if performance < 1 / 3:
        return 0

    elif performance < 2 / 3:
        return 1

    else:
        return 2


simulation_df[
    "P_level"
] = (
    simulation_df[
        "Performance_P"
    ]
    .apply(
        performance_level
    )
)

PERFORMANCE_LABELS = {
    0: "Low",
    1: "Medium",
    2: "High"
}

simulation_df[
    "P_label"
] = (
    simulation_df[
        "P_level"
    ]
    .map(
        PERFORMANCE_LABELS
    )
)

print(
    "\nPerformance-level counts:"
)

print(
    simulation_df[
        "P_label"
    ]
    .value_counts()
    .reindex(
        [
            "Low",
            "Medium",
            "High"
        ],
        fill_value=0
    )
)



# 10 Construct the complete reinforcement-learning state


simulation_df[
    "RL_State"
] = list(
    zip(
        simulation_df[
            "D_level"
        ],
        simulation_df[
            "C_level"
        ],
        simulation_df[
            "P_level"
        ]
    )
)

print(
    "\nExample RL states:"
)

print(
    simulation_df[
        [
            "Learner_Profile",
            "D_label",
            "C_label",
            "P_label",
            "RL_State"
        ]
    ].head(20)
)



# 11 Examine the distribution of the RL states


state_counts = (
    simulation_df[
        "RL_State"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nRL-state counts:"
)

print(
    state_counts
)

print(
    "\nNumber of observed RL states:",
    simulation_df[
        "RL_State"
    ]
    .nunique()
)

print(
    "Maximum possible RL states:",
    27
)



# 12 Check state coverage by learner profile


profile_state_counts = (
    simulation_df
    .groupby(
        "Learner_Profile"
    )[
        "RL_State"
    ]
    .nunique()
)

print(
    "\nNumber of observed states by learner profile:"
)

print(
    profile_state_counts
)



# 13 Check final data integrity


columns_to_check = [
    "Difficulty_D",
    "Cognitive_Effort_C",
    "Learner_Ability",
    "P_correct",
    "Correct",
    "Performance_P",
    "P_level"
]

print(
    "\nMissing values in simulation variables:"
)

print(
    simulation_df[
        columns_to_check
    ]
    .isna()
    .sum()
)

if (
    simulation_df[
        columns_to_check
    ]
    .isna()
    .any()
    .any()
):
    raise ValueError(
        "Missing values found in the synthetic learner environment."
    )


# 14 Export the synthetic learner environment


simulation_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_PATH
)

print(
    "Final shape:",
    simulation_df.shape
)



# 15 Confirm Phase 10 completion


print(
    "\nPHASE 10 COMPLETE"
)

print(
    "Input:",
    INPUT_PATH
)

print(
    "Output:",
    OUTPUT_PATH
)

print(
    "Random seed:",
    RANDOM_SEED
)

print(
    "Learner profiles:",
    LEARNER_PROFILES
)

Input file exists: True

Shape: (709, 29)

Input sample:
   Difficulty_D  Cognitive_Effort_C  D_level  C_level
0      0.635710            0.566853        2        2
1      0.443240            0.344768        1        0
2      0.284857            0.248251        0        0
3      0.434188            0.411904        1        1
4      0.482255            0.401859        1        1

Learner profiles:
{'Struggling': 0.3, 'Average': 0.55, 'Strong': 0.8}

Response-model sanity check:
Strong / low difficulty and effort P(correct) = 0.9
Average / medium difficulty and effort P(correct) = 0.426
Struggling / high difficulty and effort P(correct) = 0.057

Simulation shape: (2127, 32)

P(correct) by learner profile:
                 count      mean       std       min       25%       50%  \
Learner_Profile                                                            
Average          709.0  0.476442  0.170020  0.109565  0.338821  0.478157   
Strong           709.0  0.688315  0.150507  0.250642  0.582